# Galatiq E2E Case Simulation

Runs the same **Case Simulation Contract** as:

- CLI: `python main.py --sim=case`
- Case Lab UI: `http://localhost:8000/sim`
- pytest: `pytest tests/test_case_sim.py`

Pipeline stages: **Ingestion → Validation → Approval → Payment** (PaymentPermit-fenced).

Optional live Grok: set `XAI_API_KEY` in `.env` (see `.env.example`).

## 0. Setup & API key placeholder

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Repo root (notebook lives in notebooks/)
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")
load_dotenv(ROOT / ".env.example")  # placeholders only; real secrets override from .env

# Case brief placeholder: api_key="your_key"  model="grok-3"
# Never commit real keys — copy .env.example → .env
from invoice_automation.case_brief import GROK_MODEL, is_live_xai_key

XAI_API_KEY = os.getenv("XAI_API_KEY", "your_key")
XAI_MODEL = os.getenv("XAI_MODEL", GROK_MODEL)
USE_GROK = is_live_xai_key(XAI_API_KEY)

print("ROOT:", ROOT)
print("XAI_MODEL:", XAI_MODEL)
print("XAI_API_KEY set (live Grok):", USE_GROK)
if not USE_GROK:
    print("Using offline stub. Copy .env.example → .env and set XAI_API_KEY for Grok.")

## 1. Python CLI — Case Simulation Contract

Same entrypoint as the Galatiq case README (`python main.py ...`).

In [ ]:
cli = [
    sys.executable,
    str(ROOT / "main.py"),
    "--sim=case",
    f"--llm={'grok' if USE_GROK else 'stub'}",
    f"--work_dir={ROOT / '.galatiq_runtime' / 'notebook_cli_sim'}",
]
print("$", " ".join(cli))
proc = subprocess.run(cli, cwd=ROOT, capture_output=True, text=True)
print(proc.stdout[-4000:] if len(proc.stdout) > 4000 else proc.stdout)
if proc.stderr:
    print(proc.stderr[-1000:])
print("exit_code:", proc.returncode)
assert proc.returncode == 0, "CLI case simulation failed"

## 2. In-notebook runner (shared with pytest / Case Lab)

In [ ]:
from invoice_automation.sim.runner import run_case_simulation

work = Path(tempfile.mkdtemp(prefix="galatiq_nb_sim_"))
report = run_case_simulation(
    work,
    data_dir=ROOT / "data" / "invoices",
    use_grok=USE_GROK,
)
print(f"Pass rate: {report.passed}/{report.total}  ok={report.ok}")
print("Failures by stage:", report.fail_by_stage)
print("Terminal by stage:", report.terminal_by_stage)
assert report.ok, [r.workflow_id for r in report.failed]

In [ ]:
df = pd.DataFrame(
    [
        {
            "workflow": r.workflow_id,
            "title": r.title,
            "passed": r.passed,
            "stage": r.stage_label,
            "actual": r.actual,
            "files": ", ".join(r.files),
            "detail": r.message,
        }
        for r in report.results
    ]
)
df

## 3. Stage charts (where cases stop / fail)

In [ ]:
import matplotlib.pyplot as plt

labels = list(report.terminal_by_stage.keys())
terminals = [report.terminal_by_stage[k] for k in labels]
fails = [report.fail_by_stage.get(k, 0) for k in labels]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].barh(labels, fails, color="#b91c1c")
axes[0].set_title("Failures by pipeline stage")
axes[0].set_xlabel("count")
axes[1].barh(labels, terminals, color=["#0f172a", "#c2410c", "#a16207", "#15803d"])
axes[1].set_title("Terminal stage (all cases)")
axes[1].set_xlabel("count")
plt.tight_layout()
plt.show()

## 4. Classic pipeline smoke (single file + failure modes)

Direct `Pipeline.process_file` path (queue workers + PaymentPermit).

In [ ]:
from invoice_automation.orchestration.pipeline import Pipeline

DATA = ROOT / "data" / "invoices"
pipe = Pipeline(work_dir=work / "pipe_ok", use_grok=USE_GROK)
ok = pipe.process_file(DATA / "invoice_1001.txt")
print(ok.status, ok.issues, ok.approval_reason)
assert ok.status == "paid"

In [ ]:
cases = [
    "invoice_1002.txt",
    "invoice_1003.txt",
    "invoice_1008.txt",
    "invoice_1009.json",
]
for name in cases:
    p = Pipeline(work_dir=work / f"pipe_{name}", use_grok=False)
    r = p.process_file(DATA / name)
    print(f"{name:28} -> {r.status:10} {list(r.issues) or r.approval_reason}")

## 5. Persist report (same JSON as Case Lab)

In [ ]:
out = ROOT / ".galatiq_runtime" / "notebook_sim" / "last_report.json"
report.save(out)
print("Wrote", out)
print(json.dumps({"ok": report.ok, "passed": report.passed, "total": report.total}, indent=2))